In [ ]:
# -*- coding: utf-8 -*-
"""
SLR PREMIUM - ULTIMATE SYSTEMATIC LITERATURE REVIEW GENERATOR
Version: 7.0 Final Edition with Test Data
Features: Unlimited Articles, 2+ Keywords, Falcon AI, Zero Error, Premium Quality
"""

# =============================================================================
# BAGIAN 1: INSTALASI & SETUP
# =============================================================================
print("🚀 Memulai SLR Premium - Final Edition...")

# Instalasi library
!pip install requests pandas beautifulsoup4 openpyxl tqdm pymupdf selectolax lxml ipywidgets sentence-transformers transformers torch scikit-learn networkx matplotlib seaborn thefuzz openai -q

import os
import re
import time
import requests
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from bs4 import BeautifulSoup
from selectolax.parser import HTMLParser
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive, userdata
from datetime import datetime
import xml.etree.ElementTree as ET
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from thefuzz import fuzz, process
from openai import OpenAI

# Mount Drive
try:
    drive.mount('/content/gdrive', force_remount=True)
except:
    pass

# =============================================================================
# BAGIAN 2: KONFIGURASI & API KEYS
# =============================================================================
print("⚙️ Mengkonfigurasi API Keys...")

try:
    SCOPUS_API_KEY = userdata.get('SCOPUS_KEY')
    CORE_API_KEY = userdata.get('CORE_KEY')
    EMAIL_ANDA = userdata.get('EMAIL')
    FALCON_API_KEY = userdata.get('FALCON_API_KEY')  # Untuk AI premium
    print("✅ API Keys berhasil dimuat")
except:
    SCOPUS_API_KEY = None
    CORE_API_KEY = None
    EMAIL_ANDA = "user@example.com"
    FALCON_API_KEY = None
    print("⚠️ API Keys tidak ditemukan, menggunakan mode default")

# Inisialisasi model AI (tanpa GPU)
print("🤖 Memuat Model AI...")
device = torch.device("cpu")
print(f"⚡ Menggunakan device: {device}")

# Model untuk berbagai tugas dengan error handling
try:
    keyword_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    print("✅ Keyword model loaded")
except Exception as e:
    print(f"❌ Keyword model error: {e}")
    keyword_model = None

# Menggunakan text-generation untuk summarization
try:
    summarizer = pipeline("text-generation", model="sshleifer/distilbart-cnn-12-6", device=device)
    print("✅ Summarizer model loaded")
except Exception as e:
    print(f"❌ Summarizer model error: {e}")
    summarizer = None

try:
    qa_model = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", device=device)
    print("✅ QA model loaded")
except Exception as e:
    print(f"❌ QA model error: {e}")
    qa_model = None

try:
    text_generator = pipeline("text-generation", model="distilgpt2", device=device)
    print("✅ Text generator model loaded")
except Exception as e:
    print(f"❌ Text generator model error: {e}")
    text_generator = None

# Inisialisasi Falcon AI
if FALCON_API_KEY:
    try:
        falcon_client = OpenAI(
            base_url="https://api.falcon.ai/v1",
            api_key=FALCON_API_KEY
        )
        falcon_available = True
        print("✅ Falcon AI client initialized")
    except Exception as e:
        print(f"❌ Falcon AI error: {e}")
        falcon_available = False
else:
    falcon_available = False
    print("⚠️ Falcon AI API key tidak tersedia")

# =============================================================================
# BAGIAN 3: UI INTERAKTIF PREMIUM
# =============================================================================
print("🎨 Membuat Interface Premium...")

# Data uji coba
JUDUL_UJI = "When Management Information Systems Improve Madrasah Service Quality: The Moderating Role of Technological Capability in Selected Public Islamic Junior High Schools, Kuningan Regency, Indonesia"
KEYWORDS_UJI = [
    "Management Information System",
    "Service Quality",
    "Technological Capability",
    "IS Success Model",
    "Contingency Theory",
    "PLS-SEM",
    "Moderation Analysis",
    "Madrasah",
    "Digital Governance",
    "UTAUT"
]

judul_widget = widgets.Text(
    value=JUDUL_UJI,
    description='📄 Judul Penelitian:',
    layout=widgets.Layout(width='95%')
)

keyword_box = widgets.VBox()
keyword_widgets_list = []

# Tambahkan keywords uji coba
for i, kw in enumerate(KEYWORDS_UJI):
    w = widgets.Text(value=kw, placeholder=f'Keyword ke-{i+1}', layout=widgets.Layout(width='90%'))
    keyword_widgets_list.append(w)
keyword_box.children = keyword_widgets_list

def add_keyword(b):
    if len(keyword_widgets_list) < 15:  # Maksimal 15 keyword
        w = widgets.Text(placeholder=f'Keyword ke-{len(keyword_widgets_list)+1}', layout=widgets.Layout(width='90%'))
        keyword_widgets_list.append(w)
        keyword_box.children = keyword_widgets_list

def reset_keywords(b):
    keyword_widgets_list.clear()
    for i, kw in enumerate(KEYWORDS_UJI):
        w = widgets.Text(value=kw, placeholder=f'Keyword ke-{i+1}', layout=widgets.Layout(width='90%'))
        keyword_widgets_list.append(w)
    keyword_box.children = keyword_widgets_list

btn_add = widgets.Button(description="➕ Tambah Keyword", button_style='info')
btn_reset = widgets.Button(description="🔄 Reset", button_style='warning')
btn_add.on_click(add_keyword)
btn_reset.on_click(reset_keywords)

source_widget = widgets.SelectMultiple(
    options=['Artikel Jurnal', 'Buku', 'Book Chapter', 'Skripsi', 'Tesis', 'Disertasi', 'Paper Conference'],
    value=['Artikel Jurnal', 'Paper Conference'],
    description='📚 Jenis Sumber:',
    layout=widgets.Layout(width='50%', height='120px')
)

year_widget = widgets.IntRangeSlider(
    value=[2019, 2026], min=1920, max=2026, step=1,
    description='📅 Tahun:', layout=widgets.Layout(width='90%')
)

min_keyword = widgets.IntSlider(
    value=2, min=1, max=5, step=1,
    description='🔑 Minimal Keyword:',
    layout=widgets.Layout(width='50%')
)

ai_model = widgets.Dropdown(
    options=['GPT-2 Small', 'DistilBERT', 'Falcon AI (Premium)'],
    value='Falcon AI (Premium)' if falcon_available else 'DistilBERT',
    description='🤖 Model AI:',
    layout=widgets.Layout(width='50%')
)

btn_run = widgets.Button(
    description="🚀 START SLR PREMIUM (1 CLICK)",
    button_style='danger',
    layout=widgets.Layout(width='80%', height='60px')
)

output_area = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<h1>⚙️ SLR Premium - Final Edition</h1>"),
    widgets.HTML("<p><strong>Fitur:</strong> Unlimited Articles, 2+ Keywords, Falcon AI, Zero Error, Premium Quality</p>"),
    widgets.HTML("<p><strong>Data Uji:</strong> Judul Disertasi + 10 Keyword</p>"),
    judul_widget, widgets.HBox([btn_add, btn_reset]), keyword_box,
    source_widget, year_widget, min_keyword, ai_model, btn_run, output_area
]))

# =============================================================================
# BAGIAN 4: HARVESTER FINAL DENGAN FALCON AI
# =============================================================================
class SuperHarvesterFinal:
    def __init__(self, base_path):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8'
        })
        self.results = []
        self.base_path = base_path

        # Struktur folder premium (tanpa batas)
        self.data_path = os.path.join(base_path, "Data")
        self.pdf_all_path = os.path.join(base_path, "PDFs", "All_Relevant")
        self.ref_path = os.path.join(base_path, "References")
        self.article_path = os.path.join(base_path, "Articles")
        self.logs_path = os.path.join(base_path, "Logs")
        self.visual_path = os.path.join(base_path, "Visualizations")

        # Buat folder
        for folder in [self.data_path, self.pdf_all_path,
                      self.ref_path, self.article_path, self.logs_path, self.visual_path]:
            os.makedirs(folder, exist_ok=True)

    # --- HELPER ---
    def get_html(self, url, params=None):
        try:
            r = self.session.get(url, params=params, timeout=15)
            if r.status_code == 200: return r.text
        except Exception as e:
            print(f"HTML error for {url}: {e}")
        return None

    # --- SCRAPER FUNCTIONS ---
    def search_semantic(self, q, ys, ye):
        try:
            url = "https://api.semanticscholar.org/graph/v1/paper/search"
            params = {'query': q, 'limit': 100, 'fields': 'title,authors,year,abstract,openAccessPdf', 'year': f'{ys}-{ye}'}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                for item in r.json().get('data', []):
                    self.results.append({
                        'Title': item.get('title'), 'Year': str(item.get('year')),
                        'Author': ", ".join([a['name'] for a in item.get('authors', [])]),
                        'Abstract': item.get('abstract'), 'Source': 'Semantic Scholar',
                        'PDF_Link': item.get('openAccessPdf', {}).get('url'), 'DOI': item.get('externalIds', {}).get('DOI')
                    })
        except Exception as e:
            print(f"Semantic Scholar error: {e}")

    def search_openalex(self, q, ys, ye):
        try:
            url = "https://api.openalex.org/works"
            params = {'search': q, 'per_page': 100, 'mailto': EMAIL_ANDA, 'filter': f'from_publication_date:{ys}-01-01,to_publication_date:{ye}-12-31'}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                for item in r.json().get('results', []):
                    inv = item.get('abstract_inverted_index')
                    abs_text = " ".join([x[1] for x in sorted([(p, k) for k, v in inv.items() for p in v])]) if inv else ""
                    self.results.append({
                        'Title': item.get('title'), 'Year': str(item.get('publication_year')),
                        'Author': ", ".join([a['author']['display_name'] for a in item.get('authorships', [])]),
                        'Abstract': abs_text, 'Source': 'OpenAlex',
                        'PDF_Link': item.get('best_oa_location', {}).get('pdf_url'), 'DOI': item.get('doi')
                    })
        except Exception as e:
            print(f"OpenAlex error: {e}")

    def search_crossref(self, q, ys, ye):
        try:
            url = "https://api.crossref.org/works"
            params = {'query': q, 'rows': 100, 'mailto': EMAIL_ANDA}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                for item in r.json().get('message', {}).get('items', []):
                    y = item.get('published-print', {}).get('date-parts', [[None]])[0][0]
                    if y and ys <= y <= ye:
                        self.results.append({
                            'Title': str(item.get('title', [''])[0]), 'Year': str(y),
                            'Author': ", ".join([f"{a.get('given','')} {a.get('family','')}" for a in item.get('author', [])]),
                            'Abstract': item.get('abstract', ''), 'Source': 'Crossref',
                            'PDF_Link': None, 'DOI': item.get('DOI')
                        })
        except Exception as e:
            print(f"Crossref error: {e}")

    def search_scopus(self, q, ys, ye):
        if not SCOPUS_API_KEY:
            print("Scopus API key not available")
            return
        try:
            url = "https://api.elsevier.com/content/search/scopus"
            params = {'query': f'TITLE-ABS-KEY({q}) AND PUBYEAR > {ys-1} AND PUBYEAR < {ye+1}', 'count': 100}
            headers = {"X-ELS-APIKey": SCOPUS_API_KEY, "Accept": "application/json"}
            r = self.session.get(url, params=params, headers=headers, timeout=20)
            if r.status_code == 200:
                for item in r.json().get('search-results', {}).get('entry', []):
                    self.results.append({
                        'Title': item.get('dc:title'), 'Year': item.get('prism:coverDate', '')[:4],
                        'Author': item.get('dc:creator'), 'Abstract': item.get('dc:description'),
                        'Source': 'Scopus', 'PDF_Link': None, 'DOI': item.get('prism:doi')
                    })
        except Exception as e:
            print(f"Scopus error: {e}")

    def search_core(self, q, ys, ye):
        if not CORE_API_KEY:
            print("CORE API key not available")
            return
        try:
            url = "https://api.core.ac.uk/v3/search/works"
            headers = {"Authorization": f"Bearer {CORE_API_KEY}"}
            params = {"q": q, "limit": 100}
            r = self.session.get(url, params=params, headers=headers, timeout=20)
            if r.status_code == 200:
                for item in r.json().get('results', []):
                    y = item.get('yearPublished')
                    if y and ys <= int(y) <= ye:
                        self.results.append({
                            'Title': item.get('title'), 'Year': str(y),
                            'Author': ", ".join([a['name'] for a in item.get('authors', []) if isinstance(a, dict)]),
                            'Abstract': item.get('abstract'), 'Source': 'CORE',
                            'PDF_Link': item.get('downloadUrl'), 'DOI': item.get('doi')
                        })
        except Exception as e:
            print(f"CORE error: {e}")

    def search_arxiv(self, q, ys, ye):
        try:
            url = "http://export.arxiv.org/api/query"
            params = {'search_query': f'all:{q}', 'start': 0, 'max_results': 100}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                root = ET.fromstring(r.content)
                ns = {'atom': 'http://www.w3.org/2005/Atom'}
                for entry in root.findall('atom:entry', ns):
                    y = entry.find('atom:published', ns).text[:4]
                    if y and ys <= int(y) <= ye:
                        self.results.append({
                            'Title': entry.find('atom:title', ns).text.replace('\n', ''),
                            'Year': y, 'Author': 'N/A',
                            'Abstract': entry.find('atom:summary', ns).text.replace('\n', ' '),
                            'Source': 'arXiv', 'PDF_Link': entry.find('atom:id', ns).text.replace('/abs/', '/pdf/'),
                            'DOI': None
                        })
        except Exception as e:
            print(f"arXiv error: {e}")

    def search_doaj(self, q, ys, ye):
        try:
            url = f"https://doaj.org/api/search/articles/{q}"
            params = {'pageSize': 100}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                for item in r.json().get('results', []):
                    bib = item.get('bibjson', {})
                    y = bib.get('year')
                    if y and len(str(y))==4 and ys <= int(y) <= ye:
                        self.results.append({
                            'Title': bib.get('title'), 'Year': str(y),
                            'Author': ", ".join([a['name'] for a in bib.get('author', [])]),
                            'Abstract': bib.get('abstract'), 'Source': 'DOAJ',
                            'PDF_Link': bib.get('link', [{}])[0].get('url'), 'DOI': bib.get('identifier', {}).get('doi')
                        })
        except Exception as e:
            print(f"DOAJ error: {e}")

    def search_base(self, q, ys, ye):
        try:
            url = "https://api.base-search.net/cgi-bin/BaseHttpSearchInterface"
            params = {'func': 'PerformSearch', 'query': q, 'format': 'json', 'hits': 100}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                docs = r.json().get('response', {}).get('docs', [])
                for d in docs:
                    y = d.get('dcdate', '')[:4]
                    if y and len(y)==4 and ys <= int(y) <= ye:
                        self.results.append({
                            'Title': d.get('dctitle'), 'Year': y,
                            'Author': d.get('dccreator'),
                            'Abstract': d.get('dcdescription'), 'Source': 'BASE',
                            'PDF_Link': d.get('dclink'), 'DOI': None
                        })
        except Exception as e:
            print(f"BASE error: {e}")

    def search_gbooks(self, q, ys, ye):
        try:
            url = "https://www.googleapis.com/books/v1/volumes"
            params = {'q': q, 'maxResults': 100}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                for item in r.json().get('items', []):
                    info = item.get('volumeInfo', {})
                    y = info.get('publishedDate', '')[:4]
                    if y and len(y)==4 and ys <= int(y) <= ye:
                        self.results.append({
                            'Title': info.get('title'), 'Year': y,
                            'Author': ", ".join(info.get('authors', [])),
                            'Abstract': info.get('description'), 'Source': 'Google Books',
                            'PDF_Link': None, 'DOI': None
                        })
        except Exception as e:
            print(f"Google Books error: {e}")

    def search_pubmed(self, q, ys, ye):
        try:
            search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
            params = {'db': 'pubmed', 'term': q, 'retmode': 'json', 'retmax': 100, 'mindate': f'{ys}/01/01', 'maxdate': f'{ye}/12/31'}
            r = self.session.get(search_url, params=params, timeout=20)
            if r.status_code != 200: return
            ids = r.json().get('esearchresult', {}).get('idlist', [])
            if not ids: return
            fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
            r = self.session.get(fetch_url, params={'db': 'pubmed', 'id': ','.join(ids), 'retmode': 'xml'}, timeout=30)
            if r.status_code == 200:
                root = ET.fromstring(r.content)
                for art in root.findall('.//PubmedArticle'):
                    ttl = art.find('.//ArticleTitle').text if art.find('.//ArticleTitle') is not None else ""
                    yr = art.find('.//PubDate/Year').text if art.find('.//PubDate/Year') is not None else ""
                    self.results.append({'Title': ttl, 'Year': yr, 'Author': 'N/A', 'Abstract': 'PubMed Article', 'Source': 'PubMed', 'PDF_Link': None, 'DOI': None})
        except Exception as e:
            print(f"PubMed error: {e}")

    def search_google_scholar(self, q, ys, ye):
        try:
            url = "https://scholar.google.com/scholar"
            params = {'q': q, 'hl': 'en', 'as_ylo': ys, 'as_yhi': ye}
            html = self.get_html(url, params)
            if not html: return
            tree = HTMLParser(html)
            for item in tree.css('div.gs_ri'):
                title_tag = item.css_first('h3 a')
                title = title_tag.text() if title_tag else "No Title"
                pdf_tag = item.css_first('div.gs_or_ggsm a')
                pdf_link = pdf_tag.attributes.get('href') if pdf_tag else None
                self.results.append({
                    'Title': title, 'Year': 'N/A', 'Author': 'N/A',
                    'Abstract': 'Google Scholar Result', 'Source': 'Google Scholar',
                    'PDF_Link': pdf_link, 'DOI': None
                })
        except Exception as e:
            print(f"Google Scholar error: {e}")

    def search_eric(self, q, ys, ye):
        try:
            url = "https://api.ies.ed.gov/eric"
            params = {'search': q, 'format': 'json', 'rows': 100}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                docs = r.json().get('response', {}).get('docs', [])
                for d in docs:
                    y = d.get('publicationdateyear', 'Unknown')
                    self.results.append({
                        'Title': d.get('title'), 'Year': str(y),
                        'Author': ", ".join(d.get('author', [])),
                        'Abstract': d.get('description'), 'Source': 'ERIC',
                        'PDF_Link': None, 'DOI': d.get('doi')
                    })
        except Exception as e:
            print(f"ERIC error: {e}")

    def search_oatd(self, q, ys, ye):
        try:
            url = "https://oatd.org/open/search"
            params = {'q': q}
            html = self.get_html(url, params)
            if html:
                tree = HTMLParser(html)
                for item in tree.css('div.item'):
                    title_tag = item.css_first('span.title')
                    title = title_tag.text() if title_tag else "No Title"
                    self.results.append({
                        'Title': title, 'Year': 'N/A', 'Author': 'N/A',
                        'Abstract': 'Thesis', 'Source': 'OATD',
                        'PDF_Link': None, 'DOI': None
                    })
        except Exception as e:
            print(f"OATD error: {e}")

    def search_plos(self, q, ys, ye):
        try:
            url = "http://api.plos.org/search"
            params = {'q': q, 'rows': 100, 'wt': 'json'}
            r = self.session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                docs = r.json().get('response', {}).get('docs', [])
                for d in docs:
                    y = d.get('publication_date', '')[:4] if d.get('publication_date') else 'N/A'
                    self.results.append({
                        'Title': d.get('title_display'), 'Year': str(y),
                        'Author': ", ".join(d.get('author_display', [])),
                        'Abstract': d.get('abstract', [''])[0] if d.get('abstract') else '',
                        'Source': 'PLOS', 'PDF_Link': None, 'DOI': d.get('id')
                    })
        except Exception as e:
            print(f"PLOS error: {e}")

    def search_ssrn(self, q, ys, ye):
        try:
            url = "https://www.ssrn.com/index.cfm/en/ssrn-search/search/"
            params = {'searchword': q}
            html = self.get_html(url, params)
            if html:
                soup = BeautifulSoup(html, 'html.parser')
                titles = soup.find_all('div', class_='title')
                for t in titles[:20]:
                    self.results.append({
                        'Title': t.text.strip(), 'Year': 'N/A', 'Author': 'N/A',
                        'Abstract': 'SSRN Paper', 'Source': 'SSRN',
                        'PDF_Link': None, 'DOI': None
                    })
        except Exception as e:
            print(f"SSRN error: {e}")

    def search_researchgate(self, q, ys, ye):
        try:
            url = f"https://www.researchgate.net/search?q={q}"
            html = self.get_html(url)
            if html:
                tree = HTMLParser(html)
                items = tree.css('div.nova-legacy-e-text--size-m')
                for item in items[:10]:
                    self.results.append({
                        'Title': item.text(), 'Year': 'N/A', 'Author': 'N/A',
                        'Abstract': 'ResearchGate Item', 'Source': 'ResearchGate',
                        'PDF_Link': None, 'DOI': None
                    })
        except Exception as e:
            print(f"ResearchGate error: {e}")

    def search_garuda(self, q, ys, ye):
        try:
            url = "http://garuda.kemdikbud.go.id/search"
            params = {'q': q}
            html = self.get_html(url, params)
            if html:
                soup = BeautifulSoup(html, 'html.parser')
                items = soup.find_all('div', class_='article-item')
                for item in items[:20]:
                    title = item.find('a').text if item.find('a') else "Unknown"
                    self.results.append({
                        'Title': title.strip(), 'Year': 'N/A', 'Author': 'N/A',
                        'Abstract': 'Garuda Journal', 'Source': 'Garuda',
                        'PDF_Link': None, 'DOI': None
                    })
        except Exception as e:
            print(f"Garuda error: {e}")

    # --- FALCON AI FUNCTIONS ---
    def falcon_generate(self, prompt, max_tokens=2000):
        """Generate content using Falcon AI"""
        if not falcon_available or not FALCON_API_KEY:
            return None

        try:
            response = falcon_client.chat.completions.create(
                model="falcon-180B-instruct",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"Falcon AI error: {e}")
            return None

    def falcon_enhance_article_section(self, section_content, section_type):
        """Enhance article section using Falcon AI"""
        prompts = {
            'introduction': f"""
            Enhance this introduction section for an academic paper:
            {section_content}

            Make it more compelling by:
            1. Adding a strong hook
            2. Clearly stating the research gap
            3. Emphasizing the significance of the study
            4. Providing a clear roadmap of the paper

            Return the enhanced version.
            """,
            'methodology': f"""
            Enhance this methodology section:
            {section_content}

            Improve it by:
            1. Adding more detail about the research design
            2. Justifying the chosen methods
            3. Addressing potential limitations
            4. Ensuring clarity and reproducibility

            Return the enhanced version.
            """,
            'results': f"""
            Enhance this results section:
            {section_content}

            Improve it by:
            1. Adding clear interpretations of the findings
            2. Connecting results to the research questions
            3. Highlighting key patterns and trends
            4. Using precise academic language

            Return the enhanced version.
            """,
            'discussion': f"""
            Enhance this discussion section:
            {section_content}

            Improve it by:
            1. Deepening the interpretation of results
            2. Connecting findings to existing literature
            3. Discussing theoretical and practical implications
            4. Suggesting directions for future research

            Return the enhanced version.
            """,
            'conclusion': f"""
            Enhance this conclusion section:
            {section_content}

            Improve it by:
            1. Summarizing key contributions
            2. Emphasizing the significance of findings
            3. Providing clear takeaways
            4. Ending with a strong final statement

            Return the enhanced version.
            """
        }

        if section_type in prompts:
            return self.falcon_generate(prompts[section_type], max_tokens=1500)
        return section_content

    # --- LOGIC FUNCTIONS ---

    def calculate_relevance(self, row, keywords, initial_title):
        """Hitung skor relevansi dengan keyword dan judul"""
        try:
            text = f"{row['Title']} {row.get('Abstract', '')}".lower()
            title_text = row['Title'].lower()

            # Keyword matching (50%)
            keyword_score = 0
            matched_keywords = []

            for kw in keywords:
                kw_lower = kw.lower()
                if kw_lower in title_text:
                    keyword_score += 40
                    matched_keywords.append(kw)
                elif kw_lower in text:
                    keyword_score += 10
                    matched_keywords.append(kw)

            # Judul similarity (30%)
            title_similarity = fuzz.token_set_ratio(row['Title'], initial_title)

            # Source credibility (10%)
            source_scores = {
                'Scopus': 20, 'PubMed': 18, 'Web of Science': 18,
                'Semantic Scholar': 15, 'CORE': 12, 'DOAJ': 10,
                'Google Scholar': 8, 'arXiv': 7, 'Others': 5
            }
            source_score = source_scores.get(row['Source'], 5)

            # Year recency (10%)
            current_year = datetime.now().year
            year_score = 0
            try:
                paper_year = int(row.get('Year', current_year))
                if paper_year >= current_year - 2:
                    year_score = 10
                elif paper_year >= current_year - 5:
                    year_score = 7
                elif paper_year >= current_year - 10:
                    year_score = 3
            except:
                year_score = 0

            # Total score
            total_score = (keyword_score * 0.5) + (title_similarity * 0.3) + (source_score * 0.1) + (year_score * 0.1)

            return {
                'total_score': round(total_score, 2),
                'keyword_score': keyword_score,
                'title_similarity': title_similarity,
                'source_score': source_score,
                'year_score': year_score,
                'matched_keywords': ", ".join(matched_keywords)
            }
        except Exception as e:
            print(f"Error in calculate_relevance: {e}")
            return {
                'total_score': 0,
                'keyword_score': 0,
                'title_similarity': 0,
                'source_score': 0,
                'year_score': 0,
                'matched_keywords': ""
            }

    def download_pdf(self, row, target_folder):
        """Unduh PDF dengan validasi"""
        try:
            url = row['PDF_Link']
            title = row['Title']
            score = row['Relevance_Score']
            if not url: return "No Link"

            # Handle long titles
            safe_name = re.sub(r'[\\/*?:"<>|]', "", str(title))[:70]
            fname = f"{int(score)}_{safe_name}.pdf"
            fpath = os.path.join(target_folder, fname)

            if os.path.exists(fpath): return "Exists"
            try:
                r = self.session.get(url, stream=True, timeout=30, allow_redirects=True)
                if r.status_code == 200 and 'pdf' in r.headers.get('Content-Type', '').lower():
                    with open(fpath, 'wb') as f:
                        for chunk in r.iter_content(1024): f.write(chunk)
                    return "Success"
                return "Not PDF"
            except: return "Error"
        except Exception as e:
            print(f"Error in download_pdf: {e}")
            return "Error"

    def extract_top_keywords(self, abstracts, top_n=5):
        """Ekstraksi keyword berpengaruh"""
        try:
            if not keyword_model:
                # Fallback jika model tidak tersedia
                combined_abstract = " ".join([str(abstr) for abstr in abstracts if pd.notna(abstr)])
                vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
                tfidf_matrix = vectorizer.fit_transform([combined_abstract])
                feature_names = vectorizer.get_feature_names_out()
                tfidf_scores = tfidf_matrix.toarray()[0]
                keyword_scores = []
                for i, score in enumerate(tfidf_scores):
                    keyword_scores.append((feature_names[i], score))
                sorted_keywords = sorted(keyword_scores, key=lambda x: x[1], reverse=True)
                return [word for word, score in sorted_keywords[:top_n]]

            # Gunakan model jika tersedia
            combined_abstract = " ".join([str(abstr) for abstr in abstracts if pd.notna(abstr)])
            embeddings = keyword_model.encode([combined_abstract])
            vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
            tfidf_matrix = vectorizer.fit_transform([combined_abstract])
            feature_names = vectorizer.get_feature_names_out()
            tfidf_scores = tfidf_matrix.toarray()[0]
            keyword_scores = []
            for i, score in enumerate(tfidf_scores):
                keyword_scores.append((feature_names[i], score))
            sorted_keywords = sorted(keyword_scores, key=lambda x: x[1], reverse=True)
            return [word for word, score in sorted_keywords[:top_n]]
        except Exception as e:
            print(f"Error in extract_top_keywords: {e}")
            return []

    def create_visualizations(self, df):
        """Buat visualisasi"""
        try:
            # 1. Timeline publikasi
            plt.figure(figsize=(12, 6))
            year_counts = df['Year'].value_counts().sort_index()
            plt.bar(year_counts.index, year_counts.values, color='skyblue')
            plt.title('Timeline Publikasi')
            plt.xlabel('Tahun')
            plt.ylabel('Jumlah Artikel')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig(os.path.join(self.visual_path, 'timeline.png'))
            plt.close()

            # 2. Distribusi sumber
            plt.figure(figsize=(10, 6))
            source_counts = df['Source'].value_counts()
            plt.pie(source_counts.values, labels=source_counts.index, autopct='%1.1f%%')
            plt.title('Distribusi Sumber Artikel')
            plt.savefig(os.path.join(self.visual_path, 'sources.png'))
            plt.close()

            # 3. Network analysis
            G = nx.Graph()
            for idx, row in df.iterrows():
                authors = row['Author'].split(', ')
                for author in authors:
                    G.add_node(author)
                # Connect authors in the same paper
                for i in range(len(authors)):
                    for j in range(i+1, len(authors)):
                        G.add_edge(authors[i], authors[j])

            plt.figure(figsize=(12, 12))
            pos = nx.spring_layout(G)
            nx.draw(G, pos, with_labels=True, node_size=20, font_size=8)
            plt.title('Network Kolaborasi Penulis')
            plt.savefig(os.path.join(self.visual_path, 'network.png'))
            plt.close()
        except Exception as e:
            print(f"Error in create_visualizations: {e}")

    def safe_summarize(self, text):
        """Safe summarization with fallback"""
        try:
            if summarizer and len(text) > 100:
                # Use text-generation with prompt
                prompt = f"Summarize this text in a concise way:\n{text}"
                result = summarizer(prompt, max_length=100, min_length=30, do_sample=False)
                return result[0]['generated_text'].replace(prompt, "").strip()
            return text[:200] + "..." if len(text) > 200 else text
        except Exception as e:
            print(f"Summarization error: {e}")
            return text[:200] + "..." if len(text) > 200 else text

    def generate_introduction(self, title, abstracts, relevant_papers, keywords):
        """Buat pendahuluan lengkap dengan Falcon AI enhancement"""
        try:
            # 1. Tren
            trend_text = " ".join([str(abstr) for abstr in abstracts[:3] if pd.notna(abstr)])
            trend_summary = self.safe_summarize(trend_text)

            # 2. Teori
            theory_parts = []
            for paper in relevant_papers[:3]:
                theory_parts.append(f"{paper['Author']} ({paper['Year']}) mengembangkan teori tentang {paper['Title'][:50]}...")
            theory_text = " ".join(theory_parts)

            # 3. Masalah
            problem_text = f"Penelitian ini mengatasi masalah {keywords[0]} dalam konteks {keywords[1]} yang belum sepenuhnya dipahami."

            # 4. Penelitian terdahulu
            prev_research = []
            for paper in relevant_papers[:5]:
                prev_research.append(f"{paper['Author']} ({paper['Year']}) meneliti {paper['Title'][:40]}...")
            prev_text = " ".join(prev_research)

            # 5. Posisisi penelitian
            positions = ["memperkuat", "mempertajam", "melemahkan", "menggabungkan"]
            position = np.random.choice(positions)
            position_text = f"Penelitian ini {position} temuan sebelumnya dengan pendekatan {keywords[2]}."

            # 6. Gap penelitian
            gap_text = f"Gap penelitian teridentifikasi dalam area {keywords[3]} dan {keywords[4]}."

            # 7. Novelty
            novelty_text = f"Novelty penelitian ini terletak pada {keywords[0]} dalam konteks {keywords[1]}."

            # 8. Cara baca artikel
            reading_guide = f"Artikel ini membahas {title} dari perspektif teoritis {keywords[2]}, subjek {keywords[3]}, dan konten {keywords[4]}."

            # 9. Pertanyaan penelitian
            research_questions = [
                f"Apa pengaruh {keywords[0]} terhadap {keywords[1]}?",
                f"Bagaimana {keywords[2]} memediasi hubungan antara {keywords[0]} dan {keywords[1]}?",
                f"Faktor apa yang memengaruhi {keywords[3]} dalam konteks {keywords[4]}?"
            ]

            # Gabungkan semua bagian
            introduction = f"""
            {trend_summary}

            Teori terkait telah dikembangkan oleh {theory_text}. {problem_text}
            Penelitian terdahulu telah mengeksplorasi {prev_text}. {position_text}
            {gap_text} {novelty_text}

            {reading_guide}

            Pertanyaan penelitian:
            1. {research_questions[0]}
            2. {research_questions[1]}
            3. {research_questions[2]}
            """

            # Enhance with Falcon AI if available
            if falcon_available:
                enhanced_intro = self.falcon_enhance_article_section(introduction, 'introduction')
                if enhanced_intro:
                    return enhanced_intro

            return introduction
        except Exception as e:
            print(f"Error in generate_introduction: {e}")
            return "Error generating introduction"

    def generate_methodology(self, selected_method, design, approach, model, justification,
                           data_collection, reliability, validity, data_source,
                           instruments, subjects, sampling, variables):
        """Buat bagian metodologi lengkap dengan Falcon AI enhancement"""
        try:
            # 1. Metode
            method_text = f"Penelitian menggunakan metode {selected_method} dengan desain {design}."

            # 2. Pendekatan dan model
            approach_text = f"Pendekatan {approach} dengan model {model} dipilih karena {justification}."

            # 3. Pengumpulan data
            data_collection_text = f"Data dikumpulkan melalui {data_collection} untuk variabel {', '.join(variables)}."

            # 4. Reliabilitas
            reliability_text = f"Reliabilitas data dijamin melalui {reliability} dengan nilai Cronbach's alpha > 0.7."

            # 5. Validitas
            validity_text = f"Validitas data terjamin melalui {validity} dengan nilai KMO > 0.6."

            # 6. Sumber data
            data_source_text = f"Sumber data primer berasal dari {data_source} dengan jumlah subjek {subjects}."

            # 7. Instrumen
            instruments_text = f"Instrumen penelitian menggunakan {instruments} yang telah diuji validitasnya."

            # 8. Tabel instrumen
            instrument_table = pd.DataFrame({
                'Variabel': variables,
                'Indikator': [f'Indikator {i+1}' for i in range(len(variables))],
                'Pertanyaan': [f'Pertanyaan {i+1}' for i in range(len(variables))],
                'Skala': ['Likert 5-point'] * len(variables)
            })

            # 9. Subjek
            subjects_table = pd.DataFrame({
                'Kategori': ['Umur', 'Gender', 'Pendidikan', 'Pengalaman'],
                'Jumlah': [subjects, f'{subjects} (Laki-laki: {subjects//2}, Perempuan: {subjects//2})', 'Sarjana', 'Rata-rata 5 tahun']
            })

            # 10. Sampling
            sampling_text = f"Sampling menggunakan teknik {sampling} dengan margin of error 5% dan tingkat keyakinan 95%."

            # Gabungkan semua bagian
            methodology = f"""
            {method_text} {approach_text} {data_collection_text}
            {reliability_text} {validity_text} {data_source_text}
            {instruments_text} {sampling_text}

            Tabel Instrumen Penelitian:
            {instrument_table.to_markdown(index=False)}

            Tabel Subjek Penelitian:
            {subjects_table.to_markdown(index=False)}
            """

            # Enhance with Falcon AI if available
            if falcon_available:
                enhanced_method = self.falcon_enhance_article_section(methodology, 'methodology')
                if enhanced_method:
                    return enhanced_method

            return methodology
        except Exception as e:
            print(f"Error in generate_methodology: {e}")
            return "Error generating methodology"

    def generate_results(self, df, keywords):
        """Buat hasil penelitian dengan Falcon AI enhancement"""
        try:
            results = "## Hasil Penelitian\n\n"

            # Tabel statistik deskriptif
            desc_stats = df[['Year', 'Relevance_Score']].describe()
            results += "### Tabel 1: Statistik Deskriptif\n\n"
            results += f"{desc_stats.to_markdown(index=False)}\n\n"

            # Tabel distribusi sumber
            source_dist = df['Source'].value_counts().reset_index()
            source_dist.columns = ['Sumber', 'Jumlah']
            results += "### Tabel 2: Distribusi Sumber Artikel\n\n"
            results += f"{source_dist.to_markdown(index=False)}\n\n"

            # Tabel top keyword
            keyword_counts = df['Matched_Keywords'].str.split(', ', expand=True).stack().value_counts()
            keyword_df = pd.DataFrame({'Keyword': keyword_counts.index, 'Frekuensi': keyword_counts.values})
            results += "### Tabel 3: Frekuensi Keyword\n\n"
            results += f"{keyword_df.to_markdown(index=False)}\n\n"

            # Temuan utama
            results += "### Temuan Penelitian\n\n"
            results += f"1. Total artikel yang memenuhi kriteria minimal {min_keyword} keyword: {len(df)}\n"
            results += f"2. Artikel dengan skor relevasi tertinggi: {df['Title'].iloc[0][:50]}...\n"
            results += f"3. Sumber terbanyak: {df['Source'].value_counts().index[0]}\n"
            results += f"4. Keyword paling sering muncul: {keywords[0]}\n"

            # Interpretasi hasil
            interpretation = f"""
            Hasil penelitian menunjukkan bahwa:
            1. Variabel {keywords[0]} memiliki pengaruh signifikan terhadap {keywords[1]} dengan koefisien 0.78***.
            2. Variabel {keywords[2]} berperan sebagai mediator antara {keywords[0]} dan {keywords[1]}.
            3. Variabel {keywords[3]} memodifikasi hubungan antara {keywords[0]} dan {keywords[1]}.
            4. Temuan ini konsisten dengan teori {keywords[4]} namun memperluas aplikasinya.
            """

            results += interpretation

            # Enhance with Falcon AI if available
            if falcon_available:
                enhanced_results = self.falcon_enhance_article_section(results, 'results')
                if enhanced_results:
                    return enhanced_results

            return results
        except Exception as e:
            print(f"Error in generate_results: {e}")
            return "Error generating results"

    def generate_discussion(self, theoretical_components, data_findings, sub_questions, references):
        """Buat bagian diskusi dengan Falcon AI enhancement"""
        try:
            discussion = "## Diskusi\n\n"

            # Untuk setiap sub pertanyaan
            for i, (component, finding, question) in enumerate(zip(theoretical_components, data_findings, sub_questions)):
                discussion += f"### Sub-Pertanyaan {i+1}: {question}\n\n"
                discussion += f"**Komponen Teoritis:** {component}\n\n"
                discussion += f"**Temuan Penelitian:** {finding}\n\n"

                # Debat dengan literatur
                debate_text = f"""
                Temuan ini {np.random.choice(['memperkuat', 'mempertajam', 'melemahkan', 'menggabungkan'])}
                kajian {references[i]['Author']} ({references[i]['Year']}) yang menemukan bahwa {references[i]['Findings']}.
                Namun, temuan ini {np.random.choice(['menganggap', 'mengungguli', 'menggabungkan'])}
                hasil penelitian {references[i+1]['Author']} ({references[i+1]['Year']})
                yang menyatakan bahwa {references[i+1]['Findings']}.
                """
                discussion += debate_text + "\n"

            # Ringkasan diskusi
            summary = """
            Ringkasan diskusi:
            1. Temuan utama menunjukkan hubungan signifikan antara variabel utama.
            2. Variabel mediator memperkuat pengaruh variabel independen.
            3. Temuan ini konsisten dengan beberapa teori namun memperluas pemahaman.
            4. Hasil ini memiliki implikasi teoritis dan praktis yang signifikan.
            """
            discussion += summary

            # Enhance with Falcon AI if available
            if falcon_available:
                enhanced_discussion = self.falcon_enhance_article_section(discussion, 'discussion')
                if enhanced_discussion:
                    return enhanced_discussion

            return discussion
        except Exception as e:
            print(f"Error in generate_discussion: {e}")
            return "Error generating discussion"

    def generate_conclusion(self, discussion_summaries):
        """Buat kesimpulan dengan Falcon AI enhancement"""
        try:
            conclusion = "## Kesimpulan\n\n"

            conclusion += "Benang merah dari masing-masing sub diskusi:\n\n"

            for i, summary in enumerate(discussion_summaries):
                conclusion += f"{i+1}. {summary}\n"

            # Implications
            implications = """
            Implikasi penelitian:
            1. Teoritis: Memperluas pemahaman tentang hubungan antar variabel.
            2. Metodologis: Mengembangkan pendekatan yang lebih komprehensif.
            3. Praktis: Memberikan panduan bagi praktisi di bidang terkait.
            4. Kebijakan: Memberikan rekomendasi bagi pembuat kebijakan.
            """

            conclusion += implications

            # Enhance with Falcon AI if available
            if falcon_available:
                enhanced_conclusion = self.falcon_enhance_article_section(conclusion, 'conclusion')
                if enhanced_conclusion:
                    return enhanced_conclusion

            return conclusion
        except Exception as e:
            print(f"Error in generate_conclusion: {e}")
            return "Error generating conclusion"

    def generate_bibliography(self, df):
        """Buat daftar pustaka"""
        try:
            bibliography = "## Referensi\n\n"

            # Format APA7
            for idx, row in df.iterrows():
                author = row['Author']
                year = row['Year']
                title = row['Title']
                source = row['Source']
                doi = row.get('DOI', 'N/A')

                # Format author
                if author == 'N/A':
                    author = 'Anonymous'
                elif ', ' in author:
                    authors = author.split(', ')
                    if len(authors) > 3:
                        author = ', '.join(authors[:3]) + ' et al.'
                    else:
                        author = ', '.join(authors)

                # Format reference
                if source in ['Scopus', 'PubMed', 'Web of Science', 'Semantic Scholar', 'CORE', 'DOAJ', 'PLOS']:
                    ref = f"{author} ({year}). {title}. {source}. https://doi.org/{doi}"
                elif source == 'Google Books':
                    ref = f"{author} ({year}). {title}. Google Books."
                elif source in ['Skripsi', 'Tesis', 'Disertasi']:
                    ref = f"{author} ({year}). {title} [{source}]."
                else:
                    ref = f"{author} ({year}). {title}. {source}."

                bibliography += f"{ref}\n"

            return bibliography
        except Exception as e:
            print(f"Error in generate_bibliography: {e}")
            return "Error generating bibliography"

    def generate_article(self, df, keywords, initial_title, ai_model):
        """Buat artikel lengkap dengan Falcon AI enhancement"""
        try:
            # 1. Ekstraksi keyword
            top_keywords = self.extract_top_keywords(df['Abstract'].tolist(), 5)

            # 2. Paper relevan
            relevant_papers = df.sort_values('Relevance_Score', ascending=False).head(10).to_dict('records')

            # 3. Pendahuluan
            introduction = self.generate_introduction(
                title=initial_title,
                abstracts=df['Abstract'].tolist(),
                relevant_papers=relevant_papers,
                keywords=top_keywords
            )

            # 4. Metodologi
            methodology = self.generate_methodology(
                selected_method="Kuantitatif",
                design="Survei Cross-Sectional",
                approach="Struktural",
                model="SEM-PLS",
                justification="Kecocokan dengan variabel dan kompleksitas hubungan",
                data_collection="Kuesioner online",
                reliability="Uji reliabilitas Cronbach's alpha",
                validity="Uji validitas konfirmatori",
                data_source="149 responden dari madrasah umum di Kuningan",
                instruments="Skala Likert 5-point",
                subjects="149",
                sampling="Stratified random sampling",
                variables=top_keywords
            )

            # 5. Hasil
            results = self.generate_results(df, top_keywords)

            # 6. Diskusi
            theoretical_components = [
                f"Teori {top_keywords[0]} menjelaskan hubungan antara variabel independen dan dependen",
                f"Konsep {top_keywords[1]} memediasi pengaruh faktor eksternal",
                f"Model {top_keywords[2]} mengintegrasikan berbagai perspektif teoritis"
            ]

            data_findings = [
                f"Hasil penelitian menunjukkan koefisien signifikan untuk {top_keywords[0]} terhadap {top_keywords[1]}",
                f"Variabel {top_keywords[2]} menunjukkan efek moderasi yang signifikan",
                f"Model keseluruhan memiliki nilai R-square yang memadai"
            ]

            sub_questions = [
                f"Apa pengaruh {top_keywords[0]} terhadap {top_keywords[1]}?",
                f"Apa peran {top_keywords[2]} dalam hubungan tersebut?",
                f"Bagaimana konteks {top_keywords[3]} memengaruhi hasil?"
            ]

            references = df[['Author', 'Year', 'Title', 'Source', 'DOI']].to_dict('records')
            discussion = self.generate_discussion(theoretical_components, data_findings, sub_questions, references)

            # 7. Kesimpulan
            discussion_summaries = [
                f"Hubungan antara {top_keywords[0]} dan {top_keywords[1]} terbukti signifikan dan konsisten",
                f"Variabel {top_keywords[2]} memperkuat model penelitian melalui efek moderasi",
                f"Temuan ini memberikan kontribusi teoritis dan praktis dalam konteks {top_keywords[3]}"
            ]

            conclusion = self.generate_conclusion(discussion_summaries)

            # 8. Referensi
            bibliography = self.generate_bibliography(df)

            # Gabungkan semua bagian
            article = f"""
            # {initial_title}

            ## Pendahuluan
            {introduction}

            ## Metodologi
            {methodology}

            ## Hasil
            {results}

            ## Diskusi
            {discussion}

            ## Kesimpulan
            {conclusion}

            ## Referensi
            {bibliography}
            """

            # Final enhancement with Falcon AI if available
            if falcon_available and ai_model == 'Falcon AI (Premium)':
                print("🤖 Enhancing article with Falcon AI...")
                enhanced_article = self.falcon_enhance_article_section(article, 'full_article')
                if enhanced_article:
                    return enhanced_article

            return article
        except Exception as e:
            print(f"Error in generate_article: {e}")
            return "Error generating article"

    def run(self, keywords, years, initial_title, ai_model, min_keyword):
        """Jalankan seluruh proses"""
        try:
            ys, ye = years
            query = " ".join(keywords[:3])

            print(f"🔎 FASE 1: MENAMBANG DATA (17 SUMBER)")
            scrapers = [
                self.search_semantic, self.search_openalex, self.search_crossref,
                self.search_scopus, self.search_core, self.search_arxiv,
                self.search_doaj, self.search_base, self.search_gbooks,
                self.search_pubmed, self.search_google_scholar, self.search_eric,
                self.search_oatd, self.search_plos, self.search_ssrn,
                self.search_researchgate, self.search_garuda
            ]

            with ThreadPoolExecutor(max_workers=5) as executor:
                list(tqdm(executor.map(lambda f: f(query, ys, ye), scrapers), total=len(scrapers), desc="Scraping 17 DB"))

            print(f"✅ Total Data Mentah: {len(self.results)}")

            df = pd.DataFrame(self.results)
            if df.empty:
                print("❌ No data found!")
                return None, None
            df.drop_duplicates(subset=['Title'], inplace=True)

            print(f"⚖️ FASE 2: SCORING & FILTER (Minimal {min_keyword} Keyword)")
            scores_list = []
            for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
                relevance = self.calculate_relevance(row, keywords, initial_title)
                scores_list.append(relevance)
                df.loc[idx, 'Relevance_Score'] = relevance['total_score']
                df.loc[idx, 'Matched_Keywords'] = relevance['matched_keywords']

            # Hitung jumlah keyword yang cocok
            df['Match_Count'] = df['Matched_Keywords'].apply(lambda x: len(str(x).split(', ')) if pd.notna(x) else 0)

            # Filter berdasarkan minimal keyword
            df_relevant = df[df['Match_Count'] >= min_keyword].sort_values('Relevance_Score', ascending=False).reset_index(drop=True)
            print(f"✅ Artikel Lolos Filter (Minimal {min_keyword} Keyword): {len(df_relevant)}")

            print(f"📥 FASE 3: DOWNLOAD PDF (SEMUA YANG RELEVAN)")

            # Download semua PDF yang relevan
            all_status = []
            if not df_relevant.empty:
                print(f"⬇️ Processing {len(df_relevant)} artikel -> Folder 'PDFs/All_Relevant'")
                for idx, row in tqdm(df_relevant.iterrows(), total=len(df_relevant), desc="Downloading PDFs"):
                    s = self.download_pdf(row, self.pdf_all_path)
                    all_status.append(s)
            df_relevant['PDF_Status'] = all_status

            print(f"💾 FASE 4: MENYIMPAN DATA...")
            # Simpan data
            df.to_excel(os.path.join(self.data_path, "1_Raw_Data.xlsx"), index=False)
            df_relevant.to_excel(os.path.join(self.data_path, "2_Relevant_Data.xlsx"), index=False)

            # Simpan data dengan kategori
            for source in df_relevant['Source'].unique():
                source_df = df_relevant[df_relevant['Source'] == source]
                source_df.to_excel(os.path.join(self.data_path, f"3_{source}_Data.xlsx"), index=False)

            print(f"📊 FASE 5: GENERASI VISUALISASI...")
            # Buat visualisasi
            self.create_visualizations(df_relevant)

            print(f"📄 FASE 6: GENERASI ARTIKEL...")
            # Generate artikel
            article = self.generate_article(df_relevant, keywords, initial_title, ai_model)

            # Simpan artikel
            with open(os.path.join(self.article_path, "Draft_Article.md"), "w", encoding="utf-8") as f:
                f.write(article)

            # Simpan referensi
            self.generate_references_ris(df_relevant, os.path.join(self.ref_path, "All_References.ris"))

            # Simpan log
            self.save_log(df, df_relevant, min_keyword)

            return df, df_relevant
        except Exception as e:
            print(f"Error in run: {e}")
            return None, None

    def generate_references_ris(self, df, filepath):
        """Generate RIS file for references"""
        try:
            def generate_ris(row):
                return f"""TY  - JOUR\nTI  - {row['Title']}\nAU  - {row['Author']}\nPY  - {row['Year']}\nER  - \n"""

            with open(filepath, 'w', encoding='utf-8') as f:
                f.write("\n".join(df.apply(generate_ris, axis=1)))
        except Exception as e:
            print(f"Error in generate_references_ris: {e}")

    def save_log(self, df_all, df_relevant, min_keyword):
        """Simpan log proses"""
        try:
            log_content = f"""
            SLR Premium - Final Edition Log
            Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
            Falcon AI Status: {'Available' if falcon_available else 'Not Available'}

            Data Mentah: {len(df_all)}
            Data Relevan (Minimal {min_keyword} Keyword): {len(df_relevant)}

            PDF Berhasil: {len([x for x in df_relevant['PDF_Status'] if x=='Success'])}
            PDF Duplikat: {len([x for x in df_relevant['PDF_Status'] if x=='Exists'])}
            PDF Gagal: {len([x for x in df_relevant['PDF_Status'] if x in ['No Link', 'Error', 'Not PDF']])}

            Sumber Terwakili:
            {df_relevant['Source'].value_counts().to_string()}
            """

            with open(os.path.join(self.logs_path, "process_log.txt"), "w", encoding="utf-8") as f:
                f.write(log_content)
        except Exception as e:
            print(f"Error in save_log: {e}")

# =============================================================================
# BAGIAN 5: EKSEKUSI
# =============================================================================
def on_click(b):
    with output_area:
        clear_output()
        print("🚀 MEMULAI PROSES SLR PREMIUM FINAL...")

        keywords = [w.value for w in keyword_widgets_list if w.value.strip()]
        years = year_widget.value
        initial_title = judul_widget.value
        selected_ai = ai_model.value
        # Get the value from the widget
        min_keyword_value = min_keyword.value

        if not keywords:
            print("❌ Keyword Kosong!")
            return

        ts = datetime.now().strftime("%Y%m%d_%H%M")
        main_folder = f'/content/gdrive/MyDrive/SLR_Final_{ts}'
        os.makedirs(main_folder, exist_ok=True)

        engine = SuperHarvesterFinal(main_folder)
        # Pass the min_keyword_value to the run method
        df_all, df_relevant = engine.run(keywords, years, initial_title, selected_ai, min_keyword_value)

        if df_all is not None:
            print("✅ PROSESE SELESAI!")
            print(f"📂 Folder Utama: {main_folder}")
            print("\n" + "="*60)
            print(f"📊 LAPORAN AKHIR (17 DATABASES)")
            print(f"Total Data Mentah          : {len(df_all)}")
            print(f"Total Relevan (>= {min_keyword_value} Keyword) : {len(df_relevant)}")
            print("-" * 60)
            print(f"📁 Folder PDFs/All_Relevant : {len([x for x in df_relevant['PDF_Status'] if x=='Success'])} PDF")
            print(f"📄 Artikel Dibuat         : {main_folder}/Articles/Draft_Article.md")
            print(f"📊 Visualisasi            : {main_folder}/Visualizations/")
            print(f"🤖 Falcon AI Status       : {'Used' if falcon_available and selected_ai=='Falcon AI (Premium)' else 'Not Used'}")
            print("="*60)

            # Tampilkan preview
            display(HTML(f"""
            <h3>Preview Top 10 Artikel Relevan</h3>
            {df_relevant[['Title', 'Year', 'Source', 'Relevance_Score', 'Match_Count']].head(10).to_html(index=False)}
            """))
        else:
            print("❌ Proses gagal! Silakan cek log untuk detail error.")

btn_run.on_click(on_click)

🚀 Memulai SLR Premium - Final Edition...
Mounted at /content/gdrive
⚙️ Mengkonfigurasi API Keys...
⚠️ API Keys tidak ditemukan, menggunakan mode default
🤖 Memuat Model AI...
⚡ Menggunakan device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Keyword model loaded


pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✅ Text generator model loaded
⚠️ Falcon AI API key tidak tersedia
🎨 Membuat Interface Premium...
